# Programación I – Unidad 8_2  
## Listas avanzadas: `lambda`, `map`, `filter`, `reduce` y listas por comprensión

### Objetivos de la clase
- Introducir funciones `lambda` como funciones anónimas simples.
- Aplicar `map`, `filter` y `reduce` sobre listas de diccionarios.
- Usar **listas por comprensión** para transformar y filtrar datos.
- Comparar el enfoque **imperativo** con el enfoque **declarativo**.
- Reforzar el tema con un caso de negocio basado en el modelo entidad-relación:
  **Producto**, **Cliente** y **Venta**.

> En este notebook tomamos la idea de los archivos del año pasado y la reorientamos a un mini sistema CRUD.


## 1. Modelo de datos de la clase

Vamos a trabajar con tres estructuras, alineadas con el Excel usado en clase:

- **Producto**: `id_producto`, `categoria`, `nombre`, `precio_unitario`
- **Cliente**: `id_cliente`, `razon_social`, `email`, `cell`
- **Venta**: `id_venta`, `id_producto`, `id_cliente`, `fecha`, `cantidad`, `monto_venta`

En Python vamos a representar cada registro con un **diccionario** y cada tabla con una **lista de diccionarios**.


In [ ]:
# ============================
# Datos base (tomados del Excel)
# ============================

productos = [{'id_producto': 1, 'categoria': 'Escritura', 'nombre': 'Lápiz negro HB', 'precio_unitario': 350}, {'id_producto': 2, 'categoria': 'Escritura', 'nombre': 'Bolígrafo azul', 'precio_unitario': 500}, {'id_producto': 3, 'categoria': 'Escritura', 'nombre': 'Marcador permanente', 'precio_unitario': 1200}, {'id_producto': 4, 'categoria': 'Cuadernos', 'nombre': 'Cuaderno A4 rayado', 'precio_unitario': 3500}, {'id_producto': 5, 'categoria': 'Cuadernos', 'nombre': 'Cuaderno universitario', 'precio_unitario': 4200}, {'id_producto': 6, 'categoria': 'Organización', 'nombre': 'Carpeta N°3', 'precio_unitario': 2800}]

clientes = [{'id_cliente': 1, 'razon_social': 'YPF', 'email': 'compras@ypf.com', 'cell': '+54 9 11 3456-7890'}, {'id_cliente': 2, 'razon_social': 'Arcor', 'email': 'compras@arcor.com', 'cell': '+54 9 11 4567-1234'}, {'id_cliente': 3, 'razon_social': 'Globant', 'email': 'compras@globant.com', 'cell': '+54 9 11 5678-2345'}, {'id_cliente': 4, 'razon_social': 'MercadoLibre', 'email': 'compras@mercadolibre.com', 'cell': '+54 9 11 6789-3456'}, {'id_cliente': 5, 'razon_social': 'Techint', 'email': 'compras@techint.com', 'cell': '+54 9 11 7890-4567'}]

ventas = [{'id_venta': 1, 'id_producto': 9, 'id_cliente': 4, 'fecha': '2026-04-08', 'cantidad': 100, 'monto_venta': 950000}, {'id_venta': 2, 'id_producto': 5, 'id_cliente': 4, 'fecha': '2026-04-08', 'cantidad': 50, 'monto_venta': 210000}, {'id_venta': 3, 'id_producto': 5, 'id_cliente': 1, 'fecha': '2026-04-07', 'cantidad': 500, 'monto_venta': 2100000}]

print("Productos:")
for producto in productos:
    print(producto)

print("\nClientes:")
for cliente in clientes:
    print(cliente)

print("\nVentas:")
for venta in ventas:
    print(venta)


## 2. Repaso rápido: CRUD sobre listas de diccionarios

Antes de avanzar con `lambda`, `map`, `filter` y comprensión de listas, recordemos que seguimos trabajando sobre listas.
Eso significa que muchas operaciones CRUD todavía se resuelven recorriendo, agregando o modificando elementos.


In [ ]:
# =========================================
# Funciones CRUD simples para PRODUCTOS
# =========================================

def crear_producto(tabla_productos, id_producto, categoria, nombre, precio_unitario):
    """Crea un nuevo producto y lo agrega a la lista."""
    nuevo_producto = {
        "id_producto": id_producto,
        "categoria": categoria,
        "nombre": nombre,
        "precio_unitario": precio_unitario
    }
    tabla_productos.append(nuevo_producto)


def listar_productos(tabla_productos):
    """Muestra todos los productos."""
    for producto in tabla_productos:
        print(producto)


def buscar_producto_por_id(tabla_productos, id_producto):
    """Retorna el producto cuyo id coincida. Si no existe, devuelve None."""
    for producto in tabla_productos:
        if producto["id_producto"] == id_producto:
            return producto
    return None


def actualizar_precio_producto(tabla_productos, id_producto, nuevo_precio):
    """Actualiza el precio de un producto según su id."""
    producto = buscar_producto_por_id(tabla_productos, id_producto)
    if producto is not None:
        producto["precio_unitario"] = nuevo_precio
        return True
    return False


def eliminar_producto(tabla_productos, id_producto):
    """Elimina un producto por id."""
    for i, producto in enumerate(tabla_productos):
        if producto["id_producto"] == id_producto:
            tabla_productos.pop(i)
            return True
    return False


# Probamos el CRUD
productos_demo = productos.copy()

crear_producto(productos_demo, 99, "Oficina", "Mouse pad", 3200)
actualizar_precio_producto(productos_demo, 2, 550)
eliminar_producto(productos_demo, 6)

print("Resultado del CRUD básico:")
listar_productos(productos_demo)


## 3. Introducción a `lambda`

Una función `lambda` es una función **anónima** y breve.  
Sirve especialmente cuando necesitamos una función simple "en el momento", por ejemplo al ordenar, transformar o filtrar datos.

### Sintaxis general
```python
lambda parametros: expresion
```

### Idea clave
- Con `def`, nombramos la función y escribimos varias líneas.
- Con `lambda`, expresamos una sola operación en forma compacta.


In [ ]:
# Versión tradicional con def
def calcular_subtotal(precio_unitario, cantidad):
    return precio_unitario * cantidad

print("Subtotal con def:", calcular_subtotal(350, 3))

# Versión lambda
calcular_subtotal_lambda = lambda precio_unitario, cantidad: precio_unitario * cantidad

print("Subtotal con lambda:", calcular_subtotal_lambda(350, 3))


## 4. `lambda` aplicada a ordenamientos

En los archivos del año pasado aparecía `sorted(..., key=lambda ...)`.
Ahora lo aplicamos al contexto del negocio.

### ¿Por qué sirve?
Porque cuando ordenamos una lista de diccionarios, Python necesita saber **por qué campo** ordenar.
Eso se define con el parámetro `key`.


In [ ]:
# Ordenar productos por precio_unitario (ascendente)
productos_ordenados_precio = sorted(productos, key=lambda producto: producto["precio_unitario"])
print("Productos ordenados por precio:")
for producto in productos_ordenados_precio:
    print(producto)

print("\n" + "-"*60)

# Ordenar clientes por razon_social (descendente)
clientes_ordenados = sorted(clientes, key=lambda cliente: cliente["razon_social"], reverse=True)
print("Clientes ordenados por razón social (desc):")
for cliente in clientes_ordenados:
    print(cliente)

print("\n" + "-"*60)

# Ordenar ventas por monto_venta (descendente)
ventas_ordenadas = sorted(ventas, key=lambda venta: venta["monto_venta"], reverse=True)
print("Ventas ordenadas por monto_venta:")
for venta in ventas_ordenadas:
    print(venta)


## 5. `map`: transformar cada elemento de una colección

`map` se usa cuando queremos aplicar una misma transformación a todos los elementos de una lista.

### Forma general
```python
map(funcion, iterable)
```

Como `map` devuelve un iterador, normalmente lo convertimos a lista con `list(...)`.


In [ ]:
# Ejemplo: calcular una lista con los precios con IVA (21%)

# Enfoque imperativo
precios_con_iva = []
for producto in productos:
    precio_final = round(producto["precio_unitario"] * 1.21, 2)
    precios_con_iva.append(precio_final)

print("Precios con IVA (imperativo):", precios_con_iva)

# Enfoque declarativo con map + lambda
precios_con_iva_map = list(
    map(lambda producto: round(producto["precio_unitario"] * 1.21, 2), productos)
)

print("Precios con IVA (map):", precios_con_iva_map)


In [ ]:
# Ejemplo más cercano a CRUD:
# generar una nueva lista de productos con un campo calculado: precio_lista

productos_con_precio_lista = list(
    map(
        lambda producto: {
            "id_producto": producto["id_producto"],
            "categoria": producto["categoria"],
            "nombre": producto["nombre"],
            "precio_unitario": producto["precio_unitario"],
            "precio_lista": round(producto["precio_unitario"] * 1.21, 2)
        },
        productos
    )
)

for producto in productos_con_precio_lista:
    print(producto)


### `map` con dos listas

También podemos trabajar con dos listas relacionadas.  
Por ejemplo, si tenemos una lista de cantidades y otra de precios, podemos obtener subtotales.


In [ ]:
cantidades = [2, 4, 1, 3]
precios = [350, 500, 1200, 3500]

subtotales = list(map(lambda cantidad, precio: cantidad * precio, cantidades, precios))
print("Subtotales:", subtotales)


## 6. `filter`: quedarse solo con los elementos que cumplen una condición

`filter` se usa cuando queremos seleccionar solo algunos elementos de una colección.

### Forma general
```python
filter(funcion_booleana, iterable)
```

La función debe devolver `True` o `False`.


In [ ]:
# Ejemplo: productos cuyo precio sea mayor o igual a 1000

# Enfoque imperativo
productos_caros = []
for producto in productos:
    if producto["precio_unitario"] >= 1000:
        productos_caros.append(producto)

print("Productos caros (imperativo):")
for producto in productos_caros:
    print(producto)

print("\n" + "-"*60)

# Enfoque declarativo con filter + lambda
productos_caros_filter = list(
    filter(lambda producto: producto["precio_unitario"] >= 1000, productos)
)

print("Productos caros (filter):")
for producto in productos_caros_filter:
    print(producto)


In [ ]:
# Ejemplo: ventas con monto mayor a 500000
ventas_importantes = list(filter(lambda venta: venta["monto_venta"] > 500000, ventas))

print("Ventas importantes:")
for venta in ventas_importantes:
    print(venta)

print("\nCantidad de ventas importantes:", len(ventas_importantes))


## 7. `reduce`: combinar todos los elementos en un único resultado

`reduce` no viene incorporada como `map` y `filter`; se importa desde `functools`.

Sirve cuando queremos reducir una colección a un solo valor:
- suma total
- producto total
- máximo acumulado
- conteos acumulados


In [ ]:
from functools import reduce

# Total facturado
total_facturado = reduce(
    lambda acumulador, venta: acumulador + venta["monto_venta"],
    ventas,
    0
)

print("Total facturado:", total_facturado)

# Total de unidades vendidas
total_unidades = reduce(
    lambda acumulador, venta: acumulador + venta["cantidad"],
    ventas,
    0
)

print("Total de unidades vendidas:", total_unidades)


In [ ]:
# Buscar el nombre del cliente de cada venta y calcular
# el total vendido al cliente con id 4

ventas_cliente_4 = list(filter(lambda venta: venta["id_cliente"] == 4, ventas))

total_cliente_4 = reduce(
    lambda acumulador, venta: acumulador + venta["monto_venta"],
    ventas_cliente_4,
    0
)

print("Ventas del cliente 4:", ventas_cliente_4)
print("Total vendido al cliente 4:", total_cliente_4)


## 8. Listas por comprensión

Las listas por comprensión permiten construir nuevas listas de forma compacta.

### Forma general
```python
[expresion for elemento in iterable]
```

Y cuando hay filtro:
```python
[expresion for elemento in iterable if condicion]
```

Son una alternativa muy común a `map` y `filter`.


In [ ]:
# Transformación: obtener solo los nombres de productos
nombres_productos = [producto["nombre"] for producto in productos]
print("Nombres de productos:", nombres_productos)

# Transformación: construir etiquetas legibles
etiquetas = [
    f'{producto["nombre"]} - ${producto["precio_unitario"]}'
    for producto in productos
]
print("\nEtiquetas:")
for etiqueta in etiquetas:
    print(etiqueta)


In [ ]:
# Filtrado: productos de la categoría Escritura
productos_escritura = [
    producto
    for producto in productos
    if producto["categoria"] == "Escritura"
]

print("Productos de Escritura:")
for producto in productos_escritura:
    print(producto)

print("\n" + "-"*60)

# Filtrado + transformación:
# obtener nombres de productos con precio mayor a 1000
nombres_caros = [
    producto["nombre"]
    for producto in productos
    if producto["precio_unitario"] > 1000
]

print("Nombres de productos caros:", nombres_caros)


## 9. Comparación: imperativo vs declarativo

En esta unidad es importante mostrar que varios problemas pueden resolverse de distintas maneras.

### Enfoque imperativo
Le decimos a Python **cómo** hacer la tarea paso a paso.

### Enfoque declarativo
Le indicamos **qué** queremos obtener, usando herramientas como:
- `map`
- `filter`
- `reduce`
- comprensión de listas

No se trata de que uno siempre sea mejor que otro.  
Se trata de elegir la opción más clara para el problema.


In [ ]:
# Problema: obtener los nombres de productos cuyo precio sea >= 1000

# 1) Imperativo
resultado_1 = []
for producto in productos:
    if producto["precio_unitario"] >= 1000:
        resultado_1.append(producto["nombre"])

# 2) Filter + map
resultado_2 = list(
    map(
        lambda producto: producto["nombre"],
        filter(lambda producto: producto["precio_unitario"] >= 1000, productos)
    )
)

# 3) Lista por comprensión
resultado_3 = [
    producto["nombre"]
    for producto in productos
    if producto["precio_unitario"] >= 1000
]

print("Imperativo:", resultado_1)
print("Filter + map:", resultado_2)
print("Comprensión:", resultado_3)


## 10. Aplicación al CRUD de ventas

Ahora combinamos estas ideas con el modelo entidad-relación.  
La tabla `venta_normalizada` usa claves foráneas:

- `id_producto`
- `id_cliente`

Eso nos permite crear funciones CRUD y luego usar listas avanzadas para consultar o transformar datos.


In [ ]:
def crear_venta(tabla_ventas, id_venta, id_producto, id_cliente, fecha, cantidad, monto_venta):
    nueva_venta = {
        "id_venta": id_venta,
        "id_producto": id_producto,
        "id_cliente": id_cliente,
        "fecha": fecha,
        "cantidad": cantidad,
        "monto_venta": monto_venta
    }
    tabla_ventas.append(nueva_venta)


def buscar_venta_por_id(tabla_ventas, id_venta):
    for venta in tabla_ventas:
        if venta["id_venta"] == id_venta:
            return venta
    return None


def actualizar_cantidad_venta(tabla_ventas, id_venta, nueva_cantidad):
    venta = buscar_venta_por_id(tabla_ventas, id_venta)
    if venta is not None:
        venta["cantidad"] = nueva_cantidad
        return True
    return False


def eliminar_venta(tabla_ventas, id_venta):
    for i, venta in enumerate(tabla_ventas):
        if venta["id_venta"] == id_venta:
            tabla_ventas.pop(i)
            return True
    return False


ventas_demo = ventas.copy()

crear_venta(ventas_demo, 999, 2, 1, "2026-04-10", 10, 5000)
actualizar_cantidad_venta(ventas_demo, 2, 75)
eliminar_venta(ventas_demo, 1)

print("Ventas luego del CRUD:")
for venta in ventas_demo:
    print(venta)


In [ ]:
# Consultas útiles sobre el CRUD usando listas avanzadas

# 1) ids de productos vendidos
ids_productos_vendidos = [venta["id_producto"] for venta in ventas]
print("IDs de productos vendidos:", ids_productos_vendidos)

# 2) ventas de un cliente particular
ventas_mercadolibre = list(filter(lambda venta: venta["id_cliente"] == 4, ventas))
print("\nVentas del cliente 4:")
for venta in ventas_mercadolibre:
    print(venta)

# 3) total vendido por ese cliente
from functools import reduce
total_mercadolibre = reduce(lambda acc, venta: acc + venta["monto_venta"], ventas_mercadolibre, 0)
print("\nTotal vendido al cliente 4:", total_mercadolibre)


## 11. Mini actividades para clase

Estas actividades retoman el espíritu de los ejercicios del año pasado, pero adaptadas al contexto CRUD.

### Actividad 1 – `lambda`
1. Declarar una `lambda` que reciba un precio y una cantidad, y devuelva el subtotal.
2. Declarar una `lambda` que reciba dos productos y devuelva el que tenga mayor precio.

### Actividad 2 – `sorted` + `lambda`
1. Ordenar la lista `productos` por `precio_unitario`.
2. Crear una función que permita ordenar ascendente o descendente.

### Actividad 3 – `map`
1. Generar una nueva lista con los precios bonificados al 10%.
2. Generar una nueva lista de clientes con el email en minúsculas.

### Actividad 4 – `filter`
1. Obtener solo las ventas con `cantidad >= 100`.
2. Filtrar los clientes cuyo email pertenezca a un dominio dado.

### Actividad 5 – comprensión de listas
1. Obtener una lista con los nombres de productos de una categoría dada.
2. Obtener los ids de clientes que realizaron compras.

### Actividad 6 – `reduce`
1. Calcular el monto total vendido.
2. Calcular la cantidad total de unidades vendidas.


In [ ]:
# Espacio para resolver en clase o dejar como práctica

# Ejercicio 1: lambda subtotal
subtotal = lambda precio, cantidad: precio * cantidad
print("Subtotal:", subtotal(350, 4))

# Ejercicio 2: ordenar productos por precio descendente
productos_desc = sorted(productos, key=lambda producto: producto["precio_unitario"], reverse=True)
print("\nProductos ordenados desc:")
for producto in productos_desc:
    print(producto)


## 12. Cierre conceptual

### Ideas importantes de la unidad
- `lambda` permite definir funciones pequeñas y rápidas.
- `map` transforma elementos.
- `filter` selecciona elementos.
- `reduce` acumula o resume.
- Las **listas por comprensión** son una herramienta muy expresiva para crear nuevas listas.
- Todo esto puede aplicarse sobre estructuras tipo tabla, representadas como listas de diccionarios.
- En un contexto CRUD, estas técnicas son muy útiles para:
  - listar datos transformados,
  - filtrar registros,
  - ordenar resultados,
  - calcular métricas simples.

### Regla práctica para alumnos
- Si necesito **transformar**: pensar en `map` o comprensión.
- Si necesito **filtrar**: pensar en `filter` o comprensión.
- Si necesito **resumir** en un solo valor: pensar en `reduce`.
- Si necesito una función pequeña "al vuelo": pensar en `lambda`.
